# Task 1 — Repository Cloning & File Discovery

## Approach and reasoning

We shallow-clone the assigned repository (`--depth 1`) because the CPG only needs
the current working tree; full history would multiply the download for no
analytical gain.

After cloning we enumerate every `.py` file and apply the exclusions the lab
recommends (tests, setup files, generated code). The exclusion list is explicit
and versioned in `src/discovery/discover_files.py` rather than hard-coded inline,
so the report can state precisely what was filtered and why.

One decision that pays off later: for every included file we record a **sha256
content hash**. This gives the pipeline a free change-detection primitive that
Task 6 reuses to decide which single file to reprocess.

In [1]:
import subprocess, sys, os, json
os.chdir("..")           # notebook runs from jupyter-book/, work at repo root
print("cwd:", os.getcwd())

cwd: /home/trong/lab04


### Shallow clone (skipped if already present)

In [2]:
import os, subprocess
if not os.path.isdir("optimum"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/huggingface/optimum.git"], check=True)
commit = subprocess.run(["git", "-C", "optimum", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("cloned commit:", commit)

cloned commit: a6c775e11118d62712057bd3a8c5649898a5312d


### Run the discovery script

In [3]:
!python src/discovery/discover_files.py --repo ./optimum --out reports/file_manifest.json

Repository        : /home/trong/lab04/optimum
Total .py found   : 74
Excluded          : 15
Included (source) : 59
By top-level dir  :
    optimum              58
    docs                 1
Manifest written  -> reports/file_manifest.json


### Inspect the manifest

In [4]:
import json, pandas as pd
m = json.load(open("reports/file_manifest.json"))
print("total .py found :", m["total_py_found"])
print("excluded        :", m["excluded_count"])
print("included        :", m["included_count"])
pd.DataFrame(m["files"][:8])[["file_id", "rel_path", "size_bytes"]]

total .py found : 74
excluded        : 15
included        : 59


,file_id,rel_path,size_bytes
0,e874a4257ccdb966,docs/combine_docs.py,7264
1,a6e70c5c402cabb9,optimum/commands/base.py,5872
2,02c88dca277a3d84,optimum/commands/env.py,1940
3,f1e67e4aff2e07d0,optimum/commands/export/base.py,896
4,2987be2dec9a83cc,optimum/commands/optimum_cli.py,9660
5,ea37ff522acdd7e2,optimum/configuration_utils.py,14235
6,9fd2171637c17504,optimum/exporters/base.py,11074
7,698d13a9d0a7ffc2,optimum/exporters/error_utils.py,953


## Reflection

**What worked.** Hashing file contents during discovery rather than during
parsing meant Task 6 needed no extra machinery to detect the modified file.

**What surprised us.** `huggingface/optimum` is now a meta-package: the heavy
hardware backends (`optimum-intel`, `optimum-onnx`, ...) live in separate
repositories. The core repo is therefore small enough to parse in full, so we
dropped our original plan to sample a subset of files.

**What we would change.** The exclusion patterns are heuristic. A file named
`test_utils.py` that contains real library code would be wrongly dropped; a
stricter approach would consult the package's own build configuration.